##Creation de la Base de donnée bronze_db

In [0]:
%sql
-- Création de la base de donnée
CREATE DATABASE IF NOT EXISTS bronze_db;

-- Affiche de la base de donnée
SHOW DATABASES;

databaseName
bronze_db
default
information_schema


In [0]:
%sql
-- utilisation de la base de donnée bronze_db
USE bronze_db;
 -- selectionné la base de donnée utilisée
 SELECT current_database()

current_database()
bronze_db


In [0]:
%sql
--Supprimer la table raw_trips si elle existe en sql
--DROP TABLE IF EXISTS raw_trips;

In [0]:
%python
# voir le nombre de fichier dans le volumes nyc_taxi
display(
    dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi/bronze_data/")
      )

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/archive/,archive/,0,1793873186497
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/taxi_zone_lookup.csv,taxi_zone_lookup.csv,12331,1792662459000
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/yellow_tripdata_2024-02.parquet,yellow_tripdata_2024-02.parquet,50349284,1793873165000
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/yellow_tripdata_2024-03.parquet,yellow_tripdata_2024-03.parquet,60078280,1793873166000


In [0]:
zone_file_path = "/Volumes/workspace/default/nyc_taxi/bronze_data/taxi_zone_lookup.csv"
df_zones = spark.read.option("header", True).csv(zone_file_path)
display(df_zones)   

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
df_zones.write.format("delta").mode("ignore").saveAsTable("taxi_zone")

In [0]:
%sql
SELECT *
FROM taxi_zone
LIMIT 10;

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


### Script d'ingestion des données sources

- Définir explicitement le schema des données
- Script d'ingestion_timestamp avec les condions suivants:
- Ne crée pas de nouvelle table si la table raw_trips exist
- Ne supprime pas les anciennes données de la table raw_trips
- Ajoute une colonne ingestion_timetamp dans la table
- s'il n'ya pas de nouveaux fichier parquet, ne fait rien
- Pour ajouter de nouvelle donnée de fichier parquet, compare le timestamp de la derniere modification dans le 
- fichier parquet et la date de la derniere modification dans la table raw_trips
- Si le timestamp de la derniere modification dans le fichier parquet est plus recent que la date de la derniere 
- modification dans la table raw_trips, ajoute les nouvelles données dans la table raw_trips
- Sinon, ne fait rien
- il faut que le type des variables doit correspondre au type de la colonne dans la table raw_trips
- Si la colonne n'existe pas, il faut la créer dans la table raw_trips
- Si la colonne existe, il faut la mettre à jour dans la table raw_trips 

In [0]:

%python
from pyspark.sql.functions import current_timestamp,col,substring_index
from pyspark.sql.types import StructType,TimestampType, StringType, DoubleType, IntegerType,LongType, StructField
import datetime

# 1. Définir explicitement le schema des données
schema = StructType([
        StructField("vendorID", IntegerType(), True),
        StructField("tpep_pickup_datetime", TimestampType(), True),
        StructField("tpep_dropoff_datetime", TimestampType(), True),
        StructField("passenger_count", LongType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("RatecodeID",LongType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("payment_type", LongType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("congestion_surcharge", DoubleType(), True),
        StructField("Airport_fee", DoubleType(), True)
])

# 2. Dossier source
source_folder = "/Volumes/workspace/default/nyc_taxi/bronze_data/"
filePath = source_folder + "*.parquet"
archive_folder = source_folder + "archive/run_" + datetime.datetime.now().strftime("%Y%m%d%H%M%S") +"/"

# 3. Lister les fichiers parquet

files = [f for f in dbutils.fs.ls(source_folder) if f.name.endswith(".parquet")]

if files:
        # 4.  Lire les fichiers avec schéma forcé
        df = spark.read.schema(schema).parquet(filePath)
        #  Affichage du dataframe df
        df.show()
        # 5. Ajouter une colonne d'ingestion timestamp + une colonne nom de fichier
        df = df.withColumn("ingestion_timetamp", current_timestamp()) \
            .withColumn("file_name", substring_index(col("_metadata.file_path"),"/", -1))
       #---------------------------
       # 6. calcule le nombre de ligne du fichier
       #-----------------------------
       # Nombre de ligne dans la dataframe à ingérer
        new_rows_count = df.count()
        df.show()
        # Vrifier si la table existe dans le catalogue Spark et compter le nombre de lignes existantes dans cette table.
        if "raw_trips" in [t.name for t in spark.catalog.listTables("bronze_db")]:
                existing_rows_count = spark.table("raw_trips").count()
        else:
                existing_rows_count = 0
        # Affichage des informations
        print(f"Nombre de lignes à ingérer: {new_rows_count}")
        print(f"Nombre de lignes existantes dans la tables: {existing_rows_count}")
               
        # 7. Sauvegarde des données dans une table Delta
        df.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("raw_trips")

        # 8. Contrôle de cohérence des nombres  de lignes
        final_rows_count = spark.table("raw_trips").count()
        expected_rows = existing_rows_count + new_rows_count
        
        if final_rows_count != expected_rows:
                raise Exception(f"Le contrôle de cohérence à échoué:Le nombre de lignes dans la table final {final_rows_count} ne correspond pas aux nombres de lignes attendues {expected_rows}")       
        else:
                print(f"Le contrôle de cohérence réussi:Le nombre de lignes dans la table final {final_rows_count} est égal aux nombres de lignes attendues {expected_rows}")
        
        # 9. Archiver les fichiers parquets traités
        archive_folder = source_folder + "archive/run_" + datetime.datetime.now().strftime("%Y%m%d%H%M%S") +"/"

        for f in files:
                # deplace vers le répertoire archive
                dbutils.fs.mv(f.path, archive_folder + f.name)
                # Afficher la liste des fichiers archivés
                print(f"{len(files)}les fichiers ingérés et archivés dans{archive_folder}")

        # Afficher le contenu du dossier archive
        display(dbutils.fs.ls(archive_folder))

else:
        print("Aucun fichier parquet trouvé dans le dossier source, il n'ya rien à ingérer")  



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|vendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-03-01 00:18:51|  2024-03-01 00:23:45|              0|          1.3|         1|                 N|         142|         239|           1|        8.6|  3.5|    0.5|       2.

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/archive/run_20261105100634/yellow_tripdata_2024-02.parquet,yellow_tripdata_2024-02.parquet,50349284,1793873196000
dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/archive/run_20261105100634/yellow_tripdata_2024-03.parquet,yellow_tripdata_2024-03.parquet,60078280,1793873197000


In [0]:
%sql
SELECT count(*) 
FROM raw_trips;

count(*)
9554778


In [0]:
%sql
-- Affichage de la table raw_trips
SELECT * FROM raw_trips LIMIT 10;
 

vendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timetamp,file_name
2,2024-01-01T00:57:55.000Z,2024-01-01T01:17:43.000Z,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:03:00.000Z,2024-01-01T00:09:36.000Z,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:17:06.000Z,2024-01-01T00:35:01.000Z,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:36:38.000Z,2024-01-01T00:44:56.000Z,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:46:51.000Z,2024-01-01T00:52:57.000Z,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:54:08.000Z,2024-01-01T01:26:31.000Z,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
2,2024-01-01T00:49:44.000Z,2024-01-01T01:15:47.000Z,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
1,2024-01-01T00:30:40.000Z,2024-01-01T00:58:40.000Z,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
2,2024-01-01T00:26:01.000Z,2024-01-01T00:54:12.000Z,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet
2,2024-01-01T00:28:08.000Z,2024-01-01T00:29:16.000Z,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,2026-11-05T10:03:26.340Z,yellow_tripdata_2024-01.parquet


In [0]:
%sql
SELECT DISTINCT file_name
FROM raw_trips
ORDER BY file_name;

file_name
yellow_tripdata_2024-01.parquet
yellow_tripdata_2024-02.parquet
yellow_tripdata_2024-03.parquet


In [0]:
%sql
DESCRIBE EXTENDED raw_trips;

col_name,data_type,comment
vendorID,int,null
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
passenger_count,bigint,null
trip_distance,double,null
RatecodeID,bigint,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,bigint,null


In [0]:
%sql
--supprimer la table raw_trips
--DROP TABLE raw_trips;


In [0]:
%python
# affichage des données dans bronze_db
dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi/bronze_data/archive/")



[FileInfo(path='dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/archive/run_20261105100329/', name='run_20261105100329/', size=0, modificationTime=1793873205268),
 FileInfo(path='dbfs:/Volumes/workspace/default/nyc_taxi/bronze_data/archive/run_20261105100634/', name='run_20261105100634/', size=0, modificationTime=1793873205268)]

In [0]:
%sql
-- Renommer le nom du volume 
-- ALTER VOLUME default.bronze_files
-- RENAME TO nyc_taxi;

-----------------------
-- Path_tripdata_01 = /Volum es/workspace/default/nyc_taxi/bronze_data/yellow_tripdata_2024-01.parquet/

-- Path_zone_taxi = /Volumes/workspace/default/nyc_taxi/bronze_data/taxi_zone_lookup.csv/